# Interactive ML training

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Add here your team number teamx
team = 30

# location of your Hive database in HDFS
warehouse = "project/hive/warehouse"

spark = SparkSession.builder\
        .appName("{} - spark ML".format(team))\
        .master("yarn")\
        .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883")\
        .config("spark.sql.warehouse.dir", warehouse)\
        .config("spark.sql.avro.compression.codec", "snappy")\
        .enableHiveSupport()\
        .getOrCreate()

26/05/07 17:45:18 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.lang.reflect.Constructor.newInstance(Constructor.java:423)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServ

In [3]:
print(spark.sparkContext.uiWebUrl)
print(spark.sparkContext.applicationId)

http://hadoop-01.uni.innopolis.ru:4049
application_1777989965680_1249


In [2]:
spark.sql("USE team30_projectdb").show()
spark.sql("SHOW TABLES").show()

++
||
++
++

+----------------+--------------------+-----------+
|       namespace|           tableName|isTemporary|
+----------------+--------------------+-----------+
|team30_projectdb|   captures_bucketed|      false|
|team30_projectdb|network_connectio...|      false|
|team30_projectdb|          q1_results|      false|
|team30_projectdb|          q2_results|      false|
|team30_projectdb|          q3_results|      false|
|team30_projectdb|          q4_results|      false|
|team30_projectdb|          q5_results|      false|
|team30_projectdb|          q6_results|      false|
|team30_projectdb|          q7_results|      false|
|team30_projectdb|          q8_results|      false|
+----------------+--------------------+-----------+



In [4]:
df = spark.read.format("avro").table('team30_projectdb.network_connections_part')
df.printSchema()

root
 |-- connection_id: long (nullable = true)
 |-- capture_id: integer (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- uid: string (nullable = true)
 |-- id_orig_h: string (nullable = true)
 |-- id_orig_p: integer (nullable = true)
 |-- id_resp_h: string (nullable = true)
 |-- id_resp_p: integer (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- orig_bytes: long (nullable = true)
 |-- resp_bytes: long (nullable = true)
 |-- conn_state: string (nullable = true)
 |-- local_orig: boolean (nullable = true)
 |-- local_resp: boolean (nullable = true)
 |-- missed_bytes: long (nullable = true)
 |-- history: string (nullable = true)
 |-- orig_pkts: long (nullable = true)
 |-- orig_ip_bytes: long (nullable = true)
 |-- resp_pkts: long (nullable = true)
 |-- resp_ip_bytes: long (nullable = true)
 |-- tunnel_parents: string (nullable = true)
 |-- detailed_label: string (nullable = true)
 |-- label

In [5]:
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
import math

class CyclicalTimeTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    def __init__(self, inputCol="ts", outputCol="time_features"):
        super().__init__()
        self.inputCol = inputCol
        self.outputCol = outputCol

    def _transform(self, df):
        return df\
            .withColumn("year", F.year(F.col(self.inputCol)))\
            .withColumn("month", F.month(F.col(self.inputCol)))\
            .withColumn("day", F.dayofmonth(F.col(self.inputCol)))\
            .withColumn("hour", F.hour(F.col(self.inputCol)))\
            .withColumn("minute", F.minute(F.col(self.inputCol)))\
            .withColumn("second", F.second(F.col(self.inputCol)))\
            .withColumn("month_sin", F.sin(2 * math.pi * F.col("month") / 12))\
            .withColumn("month_cos", F.cos(2 * math.pi * F.col("month") / 12))\
            .withColumn("day_sin", F.sin(2 * math.pi * F.col("day") / 31))\
            .withColumn("day_cos", F.cos(2 * math.pi * F.col("day") / 31))\
            .withColumn("hour_sin", F.sin(2 * math.pi * F.col("hour") / 24))\
            .withColumn("hour_cos", F.cos(2 * math.pi * F.col("hour") / 24))\
            .withColumn("minute_sin", F.sin(2 * math.pi * F.col("minute") / 60))\
            .withColumn("minute_cos", F.cos(2 * math.pi * F.col("minute") / 60))\
            .withColumn("second_sin", F.sin(2 * math.pi * F.col("second") / 60))\
            .withColumn("second_cos", F.cos(2 * math.pi * F.col("second") / 60))


In [5]:
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-------------+----------+---+---+---------+---------+---------+---------+-----+--------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+--------------+-----+
|connection_id|capture_id| ts|uid|id_orig_h|id_orig_p|id_resp_h|id_resp_p|proto| service|duration|orig_bytes|resp_bytes|conn_state|local_orig|local_resp|missed_bytes|history|orig_pkts|orig_ip_bytes|resp_pkts|resp_ip_bytes|tunnel_parents|detailed_label|label|
+-------------+----------+---+---+---------+---------+---------+---------+-----+--------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+--------------+-----+
|            0|         0|  0|  0|        0|        0|        0|        0|    0|24993006|15272073|  15272073|  15272073|         0|  25011003|  25011003|           0|  25116|        0|            0|        0|            0| 

In [6]:
time_transformer = CyclicalTimeTransformer(inputCol="ts", outputCol="time_features")
df_with_time = time_transformer.transform(df)

df_with_time = df_with_time.drop("ts", "uid", "tunnel_parents")

df_with_time = df_with_time.fillna({
    "history": "unknown",
    "local_orig": False,
    "local_resp": False,
    "missed_bytes": 0,
    "service": "unknown",
    "duration": 0.0,
    "orig_bytes": 0.0,
    "resp_bytes": 0.0,
    "detailed_label": "unknown",
})

In [19]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, PCA
from pyspark.ml.linalg import Vectors

categorical_features = ["proto", "service", "conn_state", "history"]
numeric_features = [
    "duration", "orig_bytes", "resp_bytes", "missed_bytes",
    "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
    "id_orig_p", "id_resp_p", "year", "local_orig", "local_resp"
]
time_encoded_features = [
    "month_sin", "month_cos", "day_sin", "day_cos",
    "hour_sin", "hour_cos", "minute_sin", "minute_cos",
    "second_sin", "second_cos"
]

string_indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed").setHandleInvalid("skip")
    for col in categorical_features
]

one_hot_encoders = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_features
]

encoded_categorical_cols = [f"{col}_encoded" for col in categorical_features]
all_feature_cols = numeric_features + time_encoded_features + encoded_categorical_cols

vector_assembler = VectorAssembler(inputCols=all_feature_cols, outputCol="features_raw")

scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)

pca_k = 244
pca = PCA(k=pca_k, inputCol="features_scaled", outputCol="features_pca")

target_indexer = StringIndexer(inputCol="label", outputCol="label_indexed").setHandleInvalid("skip")

pipeline = Pipeline(stages=string_indexers + one_hot_encoders + [
    vector_assembler,
    scaler,
    pca,
    target_indexer
])

print("Feature engineering pipeline created")
print(f"Categorical features: {categorical_features}")
print(f"Numeric features: {numeric_features}")
print(f"Time-encoded features: {time_encoded_features}")
print(f"PCA added with k = {pca_k}")

Feature engineering pipeline created
Categorical features: ['proto', 'service', 'conn_state', 'history']
Numeric features: ['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'id_orig_p', 'id_resp_p', 'year', 'local_orig', 'local_resp']
Time-encoded features: ['month_sin', 'month_cos', 'day_sin', 'day_cos', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'second_sin', 'second_cos']
PCA added with k = 304


In [12]:
print(f"Categorical features: {categorical_features}")
print(f"Numeric features: {numeric_features}")
print(f"Time-encoded features: {time_encoded_features}")

Categorical features: ['proto', 'service', 'conn_state', 'history']
Numeric features: ['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'id_orig_p', 'id_resp_p', 'year', 'local_orig', 'local_resp']
Time-encoded features: ['month_sin', 'month_cos', 'day_sin', 'day_cos', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'second_sin', 'second_cos']


In [20]:
pipeline_model = pipeline.fit(df_with_time)
df_transformed = pipeline_model.transform(df_with_time)

In [32]:
304-60

244

In [31]:
import numpy as np

pca_model = next(stage for stage in pipeline_model.stages if stage.__class__.__name__ == "PCAModel")
explained_variance = pca_model.explainedVariance.toArray()
explained_variance_cumsum = np.cumsum(explained_variance)

print("Transformed data ready for ML")
print(f"Original feature count: {len(all_feature_cols)}")
print(f"PCA output feature count: {len(explained_variance)}")
# print(f"Total samples: {df_transformed.count()}")

# print("\nExplained variance ratio by PCA components:")
# print(explained_variance)

print("\nCumulative explained variance:")
print(explained_variance_cumsum[-60])


Transformed data ready for ML
Original feature count: 27
PCA output feature count: 304

Cumulative explained variance:
0.9029198088390817


In [34]:
df_ml = df_transformed.select(["features_pca", "label_indexed"])
df_ml = df_ml.withColumnRenamed("features_pca", "features")
df_ml = df_ml.withColumnRenamed("label_indexed", "label")

In [11]:
int(df_ml.select("features").head()[0].size)

304

In [9]:
train_data, test_data = df_ml.randomSplit([0.6, 0.4], seed=42)

print("\nClass distribution in training set:")
train_data.groupBy("label").count().show()

print("Class distribution in test set:")
test_data.groupBy("label").count().show()


Class distribution in training set:


+-----+-------+
|label|  count|
+-----+-------+
|  0.0|5269721|
|  1.0|4234505|
|  3.0|2031217|
|  2.0|3467763|
|  4.0|   5161|
|  6.0|      1|
|  5.0|   1653|
+-----+-------+

Class distribution in test set:


+-----+-------+
|label|  count|
+-----+-------+
|  0.0|3510437|
|  1.0|2820502|
|  3.0|1355024|
|  2.0|2310391|
|  4.0|   3524|
|  5.0|   1102|
|  6.0|      2|
+-----+-------+



In [10]:
%cd ~/project30

/home/team30/project30


In [11]:
import os
def run(command):
    return os.popen(command).read()

train_data.select("features", "label")\
    .write\
    .mode("overwrite")\
    .format("parquet")\
    .save("project/data/train")

# Run it from root directory of the repository
run("hdfs dfs -cat project/data/train/*.parquet > data/train.parquet")

test_data.select("features", "label")\
    .write\
    .mode("overwrite")\
    .format("parquet")\
    .save("project/data/test")

# Run it from root directory of the repository
run("hdfs dfs -cat project/data/test/*.parquet > data/test.parquet")

26/05/03 20:04:16 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 2 for reason Container marked as failed: container_e45_1747772973109_4696_01_000003 on host: hadoop-03.uni.innopolis.ru. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/05/03 20:04:16 ERROR YarnScheduler: Lost executor 2 on hadoop-03.uni.innopolis.ru: Container marked as failed: container_e45_1747772973109_4696_01_000003 on host: hadoop-03.uni.innopolis.ru. Exit status: -100. Diagnostics: Container released on a *lost* node.


''

In [3]:
train_data = spark.read.format("parquet").load("project/data/train")
test_data = spark.read.format("parquet").load("project/data/test")

root
 |-- features: struct (nullable = true)
 |    |-- type: long (nullable = true)
 |    |-- values: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- label: double (nullable = true)



In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf_classifier = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42
)

param_grid_rf = ParamGridBuilder()\
    .addGrid(rf_classifier.numTrees, [100, 150, 200])\
    .addGrid(rf_classifier.maxDepth, [5, 8, 12])\
    .addGrid(rf_classifier.minInstancesPerNode, [5, 10])\
    .addGrid(rf_classifier.maxBins, [32, 64])\
    .build()

print(f"RF total combinations: {len(param_grid_rf)}")

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedFMeasure"
)

cv_rf = CrossValidator(
    estimator=rf_classifier,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator_f1,
    numFolds=5,
    parallelism=5,
    seed=42
)

print("Starting RF kflod")
cvModel_rf = cv_rf.fit(train_data)
best_model_rf = cvModel_rf.bestModel

print(f"Best model parameters:")
for param, value in zip([p.name for p in rf_classifier.params], best_model_rf.extractParamMap().values()):
    print(f"  {param}: {value}")

RF total combinations: 36
Starting RF kflod


26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_115_2 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_110_4 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_115_0 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_2 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_110_2 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_0 !
26/05/03 23:35:15 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_110_0 !
26/05/03 23:35:15 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 1 for reason Container from a bad node: container_e46_1777835822532_0048_01_000002 on host: hadoop-03.uni.innopolis.ru. Exit status: 143. Diagnostics: [2026-05-03 23:35:15.590]Container killed on request. Exit code is 143
[2026-05-03 23:35:15

In [ ]:
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1_model1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedFMeasure")

accuracy_rf = evaluator_accuracy.evaluate(predictions_rf)
precision_rf = evaluator_precision.evaluate(predictions_rf)
recall_rf = evaluator_recall.evaluate(predictions_rf)
f1_rf = evaluator_f1_model1.evaluate(predictions_rf)

print("Model 1 (Random Forest) - Evaluation Metrics on Test Set:")
print(f"  Accuracy: {accuracy_rf:.4f}")
print(f"  Weighted Precision: {precision_rf:.4f}")
print(f"  Weighted Recall: {recall_rf:.4f}")
print(f"  Weighted F1-Score: {f1_rf:.4f}")

In [ ]:
%cd ~/project30

In [ ]:
model1_path = "project/models/model1"
best_model_rf.write().overwrite().save(model1_path)
run("hdfs dfs -get project/models/model1 models/model1")

In [ ]:
predictions_rf = best_model_rf.transform(test_data)

predictions_rf.select("label", "prediction")\
    .write\
    .mode("overwrite")\
    .format("csv")\
    .option("sep", ",")\
    .option("header","true")\
    .save("project/output/model1_predictions.csv")

run("hdfs dfs -cat project/output/model1_predictions.csv/*.csv > output/model1_predictions.csv")


In [6]:
"""
Prepare IoT network data for Spark ML by loading Hive data,
encoding time features, building the feature pipeline,
and splitting train/test data, and saving parquet-ready ML datasets.

Parquet is used for saving instead of json to persist Vector type of features
to avoid casting in future
"""

import math
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, ChiSqSelector


class CyclicalTimeTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    A PySpark Transformer that extracts cyclical time features from a timestamp column.
    This transformer takes a timestamp column as input and generates additional columns
    representing the cyclical nature of time features such as year, month, day, hour,
    minute, and second. It computes sine and cosine transformations for these features
    to capture their periodicity.
    Parameters
    ----------
    inputCol : str, optional
        The name of the input timestamp column. Default is "ts".
    outputCol : str, optional
        The name of the output column containing the generated time features.
        Default is "time_features".
    Methods
    -------
    _transform(dataset)
        Transforms the input DataFrame by adding cyclical time features.
    Returns
    -------
    DataFrame
        A DataFrame with the original columns and additional columns for year, month, day, hour,
        minute, second, and their respective sine and cosine transformations.
    """

    def __init__(self, input_col="ts", output_col="time_features"):
        super().__init__()
        self.inputCol = input_col
        self.outputCol = output_col

    def _transform(self, dataset):
        return dataset\
            .withColumn("year", F.year(F.col(self.inputCol)))\
            .withColumn("month", F.month(F.col(self.inputCol)))\
            .withColumn("day", F.dayofmonth(F.col(self.inputCol)))\
            .withColumn("hour", F.hour(F.col(self.inputCol)))\
            .withColumn("minute", F.minute(F.col(self.inputCol)))\
            .withColumn("second", F.second(F.col(self.inputCol)))\
            .withColumn("month_sin", F.sin(2 * math.pi * F.col("month") / 12))\
            .withColumn("month_cos", F.cos(2 * math.pi * F.col("month") / 12))\
            .withColumn("day_sin", F.sin(2 * math.pi * F.col("day") / 31))\
            .withColumn("day_cos", F.cos(2 * math.pi * F.col("day") / 31))\
            .withColumn("hour_sin", F.sin(2 * math.pi * F.col("hour") / 24))\
            .withColumn("hour_cos", F.cos(2 * math.pi * F.col("hour") / 24))\
            .withColumn("minute_sin", F.sin(2 * math.pi * F.col("minute") / 60))\
            .withColumn("minute_cos", F.cos(2 * math.pi * F.col("minute") / 60))\
            .withColumn("second_sin", F.sin(2 * math.pi * F.col("second") / 60))\
            .withColumn("second_cos", F.cos(2 * math.pi * F.col("second") / 60))

    def get_input_col(self):
        """
        returns the name of the input column
        """

        return self.inputCol

    def get_output_col(self):
        """
        returns the name of the output column
        """

        return self.outputCol


print("=" * 40)
print("Started ML preprocessing")


TEAM = 30
WAREHOUSE = "project/hive/warehouse"

spark = SparkSession.builder\
    .appName(f"{TEAM} - data preparation")\
    .master("yarn")\
    .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883")\
    .config("spark.sql.warehouse.dir", WAREHOUSE)\
    .config("spark.sql.avro.compression.codec", "snappy")\
    .enableHiveSupport()\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark._jsc.hadoopConfiguration().set("dfs.replication", "1")


print("Data schema")
df = spark.read.format("avro").table(
    'team30_projectdb.network_connections_part')
# df.printSchema()

# print("Null values")
# df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
#           for c in df.columns]).show()

time_transformer = CyclicalTimeTransformer(
    input_col="ts", output_col="time_features")
df_with_time = time_transformer.transform(df)

# Dropping tunnel_parents because it's all nulls
df_with_time = df_with_time.drop("ts", "uid", "tunnel_parents")
df_with_time = df_with_time.fillna({
    "history": "unknown",
    "local_orig": False,
    "local_resp": False,
    "missed_bytes": 0,
    "service": "unknown",
    "duration": 0.0,
    "orig_bytes": 0.0,
    "resp_bytes": 0.0,
    "detailed_label": "unknown",
})

cardinality = {}
for col in categorical_features:
    distinct_count = df_with_time.select(col).distinct().count()
    cardinality[col] = distinct_count

total_ohe_features = sum(cardinality.values()) - len(categorical_features)
print(f"Total OHE feats: {total_ohe_features}")


# Preprocess pipeline
categorical_features = ["proto", "service", "conn_state", "history"]
numeric_features = [
    "duration", "orig_bytes", "resp_bytes", "missed_bytes",
    "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
    "id_orig_p", "id_resp_p", "year", "local_orig", "local_resp"
]
time_encoded_features = [
    "month_sin", "month_cos", "day_sin", "day_cos",
    "hour_sin", "hour_cos", "minute_sin", "minute_cos",
    "second_sin", "second_cos"
]

string_indexers_cat = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed").setHandleInvalid("skip")
    for col in categorical_features
]

one_hot_encoders = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_features
]

categorical_assembler = VectorAssembler(
    inputCols=[f"{col}_encoded" for col in categorical_features],
    outputCol="categorical_features_raw"
)

label_indexer = StringIndexer(inputCol="label", outputCol="label_indexed").setHandleInvalid("skip")

chi_sq_selector = ChiSqSelector(
    selectorType="fpr",
    fpr=0.1,
    featuresCol="categorical_features_raw",
    outputCol="selected_categorical_features",
    labelCol="label_indexed"
)

all_vector_assembler = VectorAssembler(
    inputCols=numeric_features + time_encoded_features + ["selected_categorical_features"],
    outputCol="features_raw"
)

scaler_chi = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

pipeline = Pipeline(
    stages=string_indexers_cat + one_hot_encoders + [
    categorical_assembler,
    label_indexer,
    chi_sq_selector,
    all_vector_assembler,
    scaler
])


Started ML preprocessing
Data schema


Total OHE feats: 283


AttributeError: 'ChiSqSelector' object has no attribute 'pValues'

In [ ]:
chi_sq_selector = ChiSqSelector(
    selectorType="fpr",
    fpr=0.1,
    featuresCol="categorical_features_raw",
    outputCol="selected_categorical_features",
    labelCol="label_indexed"
)


chi_model = chi_sq_selector.fit(estimate_dataset)


In [13]:
selected = chi_model.transform(estimate_dataset)

selected.select("selected_categorical_features").head()[0].size

158

In [3]:
def estimate_df_size(df, replication=1):
    import numpy as np

    sample_size = 100000
    sample_df = df.limit(sample_size)
    

    rows = sample_df.collect()
    if not rows:
        return {"logical_bytes": 0, "replicated_bytes": 0, "logical_mb": 0, "replicated_mb": 0}
    
    import sys
    bytes_per_row = sum(sys.getsizeof(row) for row in rows) / len(rows)
    
    total_rows = df.count()
    logical_bytes = int(bytes_per_row * total_rows)
    replicated_bytes = logical_bytes * replication
    
    return {
        "logical_bytes": logical_bytes,
        "replicated_bytes": replicated_bytes,
        "logical_mb": logical_bytes / (1024**2),
        "replicated_mb": replicated_bytes / (1024**2),
        "logical_gb": logical_bytes / (1024**3),
        "replicated_gb": replicated_bytes / (1024**3),
    }

estimate_df_size(train_data)

{'logical_bytes': 960641344,
 'replicated_bytes': 960641344,
 'logical_mb': 916.1389770507812,
 'replicated_mb': 916.1389770507812,
 'logical_gb': 0.8946669697761536,
 'replicated_gb': 0.8946669697761536}

In [ ]:
train_data.select("features", "label")\
    .repartition(12)\
    .write\
    .mode("overwrite")\
    .format("parquet")\
    .save("project/data/train")

In [ ]:
test_data.select("features", "label")\
    .repartition(12)\
    .write\
    .mode("overwrite")\
    .format("parquet")\
    .save("project/data/test")